# Banking & Finance Dataset - Analysis
Libraries: **pandas, numpy, matplotlib, seaborn** (no machine learning)

Dataset: 1,000,000 transactions with customer, account, bank, channel, loan and balance details.

## 1. Imports & Settings

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)
pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:,.2f}".format)

## 2. Load Data
The first run reads the Excel file (several minutes for 1M rows). A cache file makes later runs fast.

In [ ]:
import os

path = r"D:\download\Banking & Finance Dataset — Power BI Project Guide\banking_finance_dataset_1000000_rows1.xlsx"
cache = "banking_cache.pkl"

if os.path.exists(cache):
    df = pd.read_pickle(cache)
    print("Loaded from cache")
else:
    df = pd.read_excel(path)          # needs: pip install openpyxl
    df.to_pickle(cache)
    print("Loaded from Excel and cached")

df["Transaction_Date"] = pd.to_datetime(df["Transaction_Date"], errors="coerce")
print("Shape:", df.shape)
df.head()

## 3. Data Quality

In [ ]:
print("--- Data types ---")
print(df.dtypes)
print("\n--- Missing values ---")
print(df.isna().sum())
print("\nDuplicate rows   :", df.duplicated().sum())
print("Unique customers :", df["Customer_ID"].nunique())
print("Unique accounts  :", df["Account_ID"].nunique())
print("Date range       :", df["Transaction_Date"].min(), "to", df["Transaction_Date"].max())

## 4. Descriptive Statistics

In [ ]:
num_cols = ["Transaction_Amount_INR", "Interest_Rate_Pct", "Loan_Amount_INR", "Balance_After_INR"]
df[num_cols].describe().T

In [ ]:
# NumPy statistics on Transaction Amount
amount = df["Transaction_Amount_INR"].to_numpy()
print("Mean      :", np.mean(amount))
print("Median    :", np.median(amount))
print("Std dev   :", np.std(amount))
print("90th pct  :", np.percentile(amount, 90))
print("99th pct  :", np.percentile(amount, 99))
print("Max       :", np.max(amount))

# Outliers using the IQR rule
q1, q3 = np.percentile(amount, [25, 75])
iqr = q3 - q1
outliers = ((amount < q1 - 1.5 * iqr) | (amount > q3 + 1.5 * iqr)).sum()
print(f"Outliers  : {outliers:,} ({outliers / len(df) * 100:.2f}%)")

## 5. Group Analysis

In [ ]:
def summary_by(col, top=None):
    g = (df.groupby(col)["Transaction_Amount_INR"]
           .agg(transactions="count", total="sum", average="mean")
           .sort_values("total", ascending=False))
    return g.head(top) if top else g

print("By Channel")
display(summary_by("Channel"))
print("By Transaction Type")
display(summary_by("Transaction_Type"))
print("By Account Type")
display(summary_by("Account_Type"))

In [ ]:
print("Top 10 Banks")
display(summary_by("Bank_Name", 10))
print("Top 10 Cities")
display(summary_by("Customer_City", 10))
print("Top 10 Merchant Categories")
display(summary_by("Merchant_Category", 10))

In [ ]:
status_pct = df["Transaction_Status"].value_counts(normalize=True) * 100
print("Transaction Status %")
display(status_pct.round(2))

top_customers = (df.groupby("Customer_ID")["Transaction_Amount_INR"]
                   .sum().sort_values(ascending=False).head(10))
print("Top 10 Customers by total amount")
display(top_customers)

## 6. Time Features

In [ ]:
df["Year"] = df["Transaction_Date"].dt.year
df["Month"] = df["Transaction_Date"].dt.month
df["Weekday"] = df["Transaction_Date"].dt.day_name()

monthly = df.groupby(df["Transaction_Date"].dt.to_period("M"))["Transaction_Amount_INR"].sum()
monthly.index = monthly.index.to_timestamp()

yearly = df.groupby("Year")["Transaction_Amount_INR"].agg(["count", "sum"])
display(yearly)

# Smaller sample for heavy plots
sample = df.sample(50_000, random_state=42)

## 7. Charts

### 7.1 Distributions

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 9))
for ax, col in zip(axes.ravel(), num_cols):
    sns.histplot(sample[col].dropna(), bins=40, kde=True, ax=ax)
    ax.set_title(f"Distribution of {col}")
plt.tight_layout()
plt.show()

### 7.2 Boxplots

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 5))
for ax, col in zip(axes, num_cols):
    sns.boxplot(y=sample[col], ax=ax)
    ax.set_title(col)
plt.tight_layout()
plt.show()

### 7.3 Channel and Transaction Type

In [ ]:
by_channel = summary_by("Channel")
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.barplot(x=by_channel.index, y=by_channel["total"], ax=axes[0])
axes[0].set_title("Total Amount by Channel")
axes[0].tick_params(axis="x", rotation=30)

sns.countplot(data=df, x="Transaction_Type",
              order=df["Transaction_Type"].value_counts().index, ax=axes[1])
axes[1].set_title("Transactions by Type")
axes[1].tick_params(axis="x", rotation=30)
plt.tight_layout()
plt.show()

### 7.4 Top Banks and Cities

In [ ]:
tb = summary_by("Bank_Name", 10)
tc = summary_by("Customer_City", 10)

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
sns.barplot(x=tb["total"], y=tb.index, ax=axes[0])
axes[0].set_title("Top 10 Banks by Total Amount")
sns.barplot(x=tc["total"], y=tc.index, ax=axes[1])
axes[1].set_title("Top 10 Cities by Total Amount")
plt.tight_layout()
plt.show()

### 7.5 Transaction Status

In [ ]:
plt.figure(figsize=(7, 7))
plt.pie(status_pct.values, labels=status_pct.index, autopct="%1.1f%%", startangle=90)
plt.title("Transaction Status Share")
plt.show()

### 7.6 Monthly Trend

In [ ]:
plt.figure(figsize=(14, 5))
plt.plot(monthly.index, monthly.values, marker="o")
plt.title("Monthly Total Transaction Amount")
plt.xlabel("Month")
plt.ylabel("Amount (INR)")
plt.show()

### 7.7 Weekday Pattern

In [ ]:
order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
wk = df.groupby("Weekday")["Transaction_Amount_INR"].count().reindex(order)

plt.figure(figsize=(10, 5))
sns.barplot(x=wk.index, y=wk.values)
plt.title("Transactions by Weekday")
plt.xticks(rotation=30)
plt.show()

### 7.8 Heatmap: Channel vs Transaction Type

In [ ]:
pivot = df.pivot_table(index="Channel", columns="Transaction_Type",
                       values="Transaction_Amount_INR", aggfunc="mean")
plt.figure(figsize=(11, 6))
sns.heatmap(pivot, annot=True, fmt=",.0f", cmap="YlGnBu")
plt.title("Average Transaction Amount: Channel vs Transaction Type")
plt.show()

### 7.9 Amount by Account Type

In [ ]:
plt.figure(figsize=(10, 5))
sns.boxplot(data=sample, x="Account_Type", y="Transaction_Amount_INR", showfliers=False)
plt.title("Transaction Amount by Account Type")
plt.show()

### 7.10 Correlation

In [ ]:
plt.figure(figsize=(8, 6))
sns.heatmap(sample[num_cols].corr(), annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Correlation Matrix")
plt.show()

### 7.11 Amount vs Balance

In [ ]:
plt.figure(figsize=(9, 6))
sns.scatterplot(data=sample.sample(5000, random_state=1),
                x="Transaction_Amount_INR", y="Balance_After_INR", alpha=0.4)
plt.title("Transaction Amount vs Balance After")
plt.show()

## 8. Key Findings

In [ ]:
print("=" * 55)
print("KEY FINDINGS")
print("=" * 55)
print(f"Total transactions   : {len(df):,}")
print(f"Total amount (INR)   : {df['Transaction_Amount_INR'].sum():,.0f}")
print(f"Average transaction  : {df['Transaction_Amount_INR'].mean():,.2f}")
print(f"Top channel          : {summary_by('Channel').index[0]}")
print(f"Top bank             : {summary_by('Bank_Name').index[0]}")
print(f"Top city             : {summary_by('Customer_City').index[0]}")
print(f"Most common status   : {status_pct.index[0]} ({status_pct.iloc[0]:.1f}%)")
print("=" * 55)